# <center>[📝 Достаем текстовые эмбеддинги](https://stepik.org/lesson/825509/step/5?unit=829016)</center>

### Оглавление ноутбука

<img src='../images/bert.jpg' align="right" width="450" height="450" />
<br>

<p><font size="3" face="Arial" font-size="large"><ul type="square">
    
<li><a href="#c1">👁 Подключаемся к гугл-диску и GPU</a></li>
<li><a href="#c2">🗜 Импортируем библиотеки</a></li>
<li><a href="#c3">📩 Считываем данные</a></li>
<li><a href="#c4">🤗 Делаем предсказания языковой моделью</a></li>
<li><a href="#c5">🍢 Усредняем эмбеддинги</a></li>
<li><a href="#c6">🧵 Объединяем все в класс</a></li>
<li><a href="#c7">🗣 Советы по использованию языковых моделей</a>

</li></ul></font></p>

<div class="alert alert-info">
Вы уже наверняка слышали о базовых способах работы с текстами: `tf-idf`, наивный байес, `word2vec`. Однако, большие предобученные языковые модели работают в тысячу раз лучше (думаю, вы уже слышали про `ChatGpt`). Поэтому, в соревнованиях при работе с текстом, обычно применяются именно они. И сегодня мы рассмотрим, как можно достать эмбеддинги текстовых признаков при помощи таких языковых моделей, чтобы потом использовать их в качестве признаков.

## 👁 Подключаемся к гугл-диску
<p id="c1"></p>   

<div class="alert alert-info"> 
    
Языковые модели, которые мы будем рассматривать, очень большие и поэтому, для того, чтобы ими пользоваться и это не занимало вечность, нам нужно будет использовать графические ускорители (GPU). Для этого, воспользуемся сервисом [google colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), который как раз предоставляет такие возможности. Первым делом подключимся к нашему гугл-диску и проверим, что GPU на месте.

In [18]:
# from google.colab import drive
# drive.mount('/content/drive')


In [5]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126


Looking in indexes: https://download.pytorch.org/whl/cu126


In [6]:
import torch
from torch import nn

# If there's a GPU available...
if torch.cuda.is_available():    

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
We will use the GPU: NVIDIA GeForce RTX 4080 SUPER


## 🗜 Импортируем библиотеки
<p id="c2"></p>   

In [7]:
!pip install transformers sentencepiece sentence-transformers simpletransformers -q 

In [8]:
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments, default_data_collator, DebertaV2Tokenizer, PegasusForConditionalGeneration, PegasusTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
import torch 

import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm

## 📩 Считываем данные
<p id="c3"></p>   

In [9]:
data = pd.read_csv('../data/text_classification_train_only_text.csv')
data.head()

,text
0,Ледник Пасторури это цирковой ледник расположе...
1,Главные участники предстоящего Betokenoid 274 ...
2,Ttokenoid Btokenoid – карта с которой можно не...
3,В Сильверстоуне произошли крупные обновления а...
4,На протяжении более чем 30 лет Вестсайд являет...


In [10]:
data['text'] = data['text'].astype(str)
data.shape

(7500, 1)

## 🤗 Делаем предсказания языковой моделью
<p id="c4"></p>   

<div class="alert alert-info">

Для того, чтобы проинференсить, нам понадобятся две сущности: `модель` и `токенайзер`. Токенайзер будет предобрабатывать текст, чтобы его можно было подать в модель (разбивать на токены, считать `attention_mask`, обрезать лишний текст и тд), в то время как модель будет непосредственно считать сами эмбеддинги.

In [11]:
model_name = 'sberbank-ai/ruBert-base'
max_len = 512

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).cuda()

c:\Users\b_jumaev\Documents\vscode\Competitive_Data_Science\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\b_jumaev\.cache\huggingface\hub\models--sberbank-ai--ruBert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18113.03it/s]
BertMo

In [12]:
sentence = data['text'][0]
encoded_input = tokenizer([sentence], # список текстов, которые хотим прогнать
                          padding='max_length', # будем дополнять токены нулями до max_len 
                          truncation=True, # если слишком много токенов, то будем удалять последние
                          max_length=max_len, # максимальная длина текстов, с которой может работать модель
                          return_tensors='pt') # возвращать будем тензоры, чтобы было удобно работать
with torch.no_grad():
    hidden_state, cls_head = model(input_ids=encoded_input['input_ids'].cuda(), return_dict=False)
    # hidden_state - выходы модели для каждого токена, shape = (batch_size, max_len, emb_size)
    # cls_head - выход модели для токена [CLS], в котором аккумулируется информация о всем предложение, shape = (batch_size, emb_size)

In [13]:
hidden_state.shape, cls_head.shape

(torch.Size([1, 512, 768]), torch.Size([1, 768]))

## 🍢 Усредняем эмбеддинги
<p id="c5"></p>   

<div class="alert alert-info">

Для того, чтобы посчитать общий эмбеддинг предложения, можно либо взять информацию из `cls_head`, либо усреднить `hidden_state` по всем словам. Давайте посчитаем усредненный `hidden_state`.

In [14]:
# attention_mask - содержит 1, если на этой позиции стоит осмысленный токен и 0, если это padding
attention_mask = encoded_input['attention_mask']
token_embeddings = hidden_state.detach().cpu()
attention_mask[0][:50], token_embeddings[0][0][:50]

(tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]),
 tensor([ 0.3078,  0.4664,  0.7443,  0.8290, -0.4497,  0.3474, -0.6934,  0.3632,
          0.2229, -1.0116, -0.1834,  0.0818, -0.0139, -0.3390,  0.6213, -0.4389,
         -0.0878,  1.0532, -1.4599, -0.1087, -0.4355,  0.1825,  0.3825, -0.1280,
          0.5062, -0.0472,  0.3235,  0.3271, -0.5110,  0.2902,  0.3373, -0.0867,
         -0.5511,  1.3511,  0.0597,  1.2196, -0.1140, -0.3848,  0.1512,  0.1980,
         -1.9379,  0.0468,  0.9116, -0.3323,  1.1896,  0.0047, -0.0728,  0.0836,
          0.1591,  0.3771]))

In [15]:
token_embeddings.shape, attention_mask.shape

(torch.Size([1, 512, 768]), torch.Size([1, 512]))

In [16]:
attention_mask.unsqueeze(-1).shape

torch.Size([1, 512, 1])

In [17]:
# растянем нашу маску так, чтобы ее размер совпадал с размером hidden_state
input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
input_mask_expanded.shape, sum_embeddings.shape

(torch.Size([1, 512, 768]), torch.Size([1, 768]))

In [18]:
attention_mask.sum()

tensor(34)

In [19]:
(sum_embeddings / attention_mask.sum()).shape

torch.Size([1, 768])

## 🧵 Объединяем все в класс
<p id="c6"></p>   

In [20]:
class TextEmbeddings:
    def __init__(self, add_cls_embeddings=True, add_mean_embeddings=False):
        self.add_mean_embeddings = add_mean_embeddings
        self.add_cls_embeddings = add_cls_embeddings
        if add_cls_embeddings is False and add_mean_embeddings is False:
            raise 'Error: you should select at least one type of embeddings to be computed'

    def mean_pooling(self, hidden_state, attention_mask):
        """
        Возвращает усредненный с учетом attention_mask hidden_state.
        """
        token_embeddings = hidden_state.detach().cpu() 
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        return sum_embeddings / attention_mask.sum()

    def extract_embeddings(self, texts, model_name, max_len):
        """
        Возвращает значения, посчитанные данной моделью - эмбеддинги для всех текстов из texts.
        """
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).cuda()
        text_features = []
        for sentence in tqdm(texts):
            encoded_input = tokenizer([sentence],
                                      padding='max_length',
                                      truncation=True,
                                      max_length=max_len,
                                      return_tensors='pt')
            with torch.no_grad():
                hidden_state, cls_head = model(input_ids=encoded_input['input_ids'].cuda(), return_dict=False)
                sentence_embeddings = self.mean_pooling(hidden_state, encoded_input['attention_mask'])
            
            now_emb = []
            if self.add_cls_embeddings:
                now_emb.append(cls_head.detach().cpu().numpy().flatten())
            
            if self.add_mean_embeddings:
                now_emb.append(sentence_embeddings.detach().cpu().numpy().flatten())
            
            text_features.append(np.concatenate(now_emb, axis=0))
        return text_features

    def add_many_embeddings(self, df, text_col, models):
        """"
        Добавляет в качестве признаков эмбеддинги для колонки text_col.
        В качестве моделей и максимальных длин используются models.
        """
        for model_name, max_len in models:
            print(model_name)
            text_features = self.extract_embeddings(df[text_col], model_name, max_len)
            text_features_df = pd.DataFrame(text_features, columns = [f'{model_name}_{text_col}_feature_{i}' for i in range(len(text_features[0]))])
            df = df.join(text_features_df)
            df.to_csv('transformers_text_features.csv', index=False)
            os.system('cp /content/transformers_text_features.csv /content/drive/MyDrive/datasets/transformers_text_features.csv')
        return df

In [21]:
# Полный список поддерживаемых моделей можно найти на https://huggingface.co/models
models = [
          ('cointegrated/LaBSE-en-ru', 512),
        #   ('sberbank-ai/ruRoberta-large', 512),
        #   ('sberbank-ai/sbert_large_nlu_ru', 512),
        #   ('sberbank-ai/sbert_large_mt_nlu_ru', 512),
        #   ('sberbank-ai/ruBert-large', 512),
          ('sberbank-ai/ruBert-base', 512),
          ('cointegrated/rubert-tiny2', 2048),
          ('DeepPavlov/rubert-base-cased-conversational', 512),
        #   ('microsoft/mdeberta-v3-base', 512),
        #   ('vicgalle/xlm-roberta-large-xnli-anli', 512),
        #   ('MoritzLaurer/mDeBERTa-v3-base-mnli-xnli', 512),
        #   ('facebook/bart-large-mnli', 1024)
]

In [24]:
# text_embeddings = TextEmbeddings(True, True)
data = text_embeddings.add_many_embeddings(data, 'text', models)

cointegrated/LaBSE-en-ru


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12423.59it/s]
BertModel LOAD REPORT from: cointegrated/LaBSE-en-ru
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [23]:
data.shape

(7500, 1)

In [20]:
data.head()

,text,cointegrated/LaBSE-en-ru_text_feature_0,cointegrated/LaBSE-en-ru_text_feature_1,cointegrated/LaBSE-en-ru_text_feature_2,cointegrated/LaBSE-en-ru_text_feature_3,cointegrated/LaBSE-en-ru_text_feature_4,cointegrated/LaBSE-en-ru_text_feature_5,cointegrated/LaBSE-en-ru_text_feature_6,cointegrated/LaBSE-en-ru_text_feature_7,cointegrated/LaBSE-en-ru_text_feature_8,...,DeepPavlov/rubert-base-cased-conversational_text_feature_1526,DeepPavlov/rubert-base-cased-conversational_text_feature_1527,DeepPavlov/rubert-base-cased-conversational_text_feature_1528,DeepPavlov/rubert-base-cased-conversational_text_feature_1529,DeepPavlov/rubert-base-cased-conversational_text_feature_1530,DeepPavlov/rubert-base-cased-conversational_text_feature_1531,DeepPavlov/rubert-base-cased-conversational_text_feature_1532,DeepPavlov/rubert-base-cased-conversational_text_feature_1533,DeepPavlov/rubert-base-cased-conversational_text_feature_1534,DeepPavlov/rubert-base-cased-conversational_text_feature_1535
0,Ледник Пасторури это цирковой ледник расположе...,-0.076881,-0.412762,-0.056001,0.046433,-0.449512,-0.253623,-0.301378,-0.103132,-0.002914,...,-0.783512,0.589172,-0.022016,-0.231998,0.875344,-0.112135,-0.048334,0.161959,0.981492,-0.298606
1,Главные участники предстоящего Betokenoid 274 ...,-0.046052,-0.103519,-0.111545,-0.114912,0.039759,-0.388762,-0.350188,-0.019205,-0.062527,...,-0.662196,0.968559,-0.271811,0.215214,1.008041,-0.207437,0.750243,0.188331,0.905673,-0.198813
2,Ttokenoid Btokenoid – карта с которой можно не...,0.070633,-0.161732,0.134272,0.207949,0.076965,0.045200,-0.030984,-0.070647,0.421526,...,-0.759898,0.173297,-0.612715,-0.515750,0.493289,-0.486974,0.029368,1.051441,0.681879,-1.202563
3,В Сильверстоуне произошли крупные обновления а...,-0.181576,-0.094991,0.289999,0.184837,0.076356,-0.125467,0.171146,-0.018428,-0.055698,...,-0.697004,0.126596,-0.838320,-0.295630,1.008091,0.097093,-0.005237,0.303587,0.788490,-0.606933
4,На протяжении более чем 30 лет Вестсайд являет...,-0.077017,-0.280559,0.147400,0.345588,-0.185278,-0.234000,-0.023668,-0.418108,-0.309198,...,-0.526098,0.874267,-0.634379,0.069256,0.950485,-0.208338,0.473490,0.558341,1.220465,-1.187832


## 🗣 Советы по использованию языковых моделей
<p id="c7"></p>   

<div class="alert alert-info">
Напоследок, несколько советов по работе с текстами.
    
**Когда применять языковые модели:**
1) Тексты не очень большие (в идеале до 512 токенов)
2) Вы хотите опираться на смысл текста, а не на ключевые слова
3) Есть GPU ресурсы и от вас не требуют мгновенной скорости работы
    
**Как выбрать какие модели использовать:**
1) Лучше всего использовать модели, предобученные на вашем домене, если такие есть
2) Обратите внимание, что ваша модель работает с необходимым вам языком
3) Если очень хочется, то тексты можно перевести на английский (см. урок про парсинг) и использовать модели уже для переведнных текстов 
4) Модели можно дообучать на вашей задаче
5) Можно доставать эмбеддинги и использовать их в качестве признаков, а можно просто предсказывать таргет с помощью языковой модели
6) Если есть время и мощности, то не пренебрегайте стекингом разных языковых моделей

In [25]:
model_name = 'vicgalle/xlm-roberta-large-xnli-anli'
max_len = 512
sentence = 'Привет, чемпион!'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).cuda()

c:\Users\b_jumaev\Documents\vscode\Competitive_Data_Science\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\b_jumaev\.cache\huggingface\hub\models--vicgalle--xlm-roberta-large-xnli-anli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 12299.

In [31]:
sentence

'Привет, чемпион!'

In [32]:
encoded_input = tokenizer([sentence], # список текстов, которые хотим прогнать
                          padding='max_length', # будем дополнять токены нулями до max_len 
                          truncation=True, # если слишком много токенов, то будем удалять последние
                          max_length=max_len, # максимальная длина текстов, с которой может работать модель
                          return_tensors='pt') # возвращать будем тензоры, чтобы было удобно работать
# with torch.no_grad():
#     hidden_state, cls_head = model(input_ids=encoded_input['input_ids'].cuda(), return_dict=False)
#     # hidden_state - выходы модели для каждого токена, shape = (batch_size, max_len, emb_size)
#     # cls_head - выход модели для токена [CLS], в котором аккумулируется информация о всем предложение, shape = (batch_size, emb_size)

In [33]:
encoded_input['input_ids'].shape, encoded_input['attention_mask'].shape

(torch.Size([1, 512]), torch.Size([1, 512]))

In [34]:
# Задача крайне проста: воспроизведите у себя код, который показывался на семинаре и посчитайте input_ids (токены текста после применения токенайзера) для текста: "Привет, чемпион!". В качестве токенайзера используйте 'vicgalle/xlm-roberta-large-xnli-anli' с max_len=512.

# В качестве ответа введите сумму по всему input_ids.

input_ids_sum = encoded_input['input_ids'].sum().item()
input_ids_sum

165688